## Docker lab assignment

Welcome to this lab where you'll apply your Docker skills. Remember, Docker is a powerful tool for creating, deploying, and running applications in containers. These containers encapsulate an application along with its dependencies, ensuring consistent runs across different environments.

This repository includes both backend and frontend applications. Our backend, similar to the FastAPI app developed in previous labs, uses the Stable Diffusion API to create images from user inputs. The frontend is a straightforward Streamlit app for interacting with the backend.


**Our primary objective is to construct a Docker Compose file that runs both applications in a containerized setup.**


First, we'll examine our FastAPI backend and conduct some tests. Then, we'll explore the Streamlit frontend, observing how the two applications interact. After testing both applications, we'll begin the process of dockerizing them individually before creating a Docker Compose file for their joint operation.

Let's get started!

## Backend App
The backend is a basic FastAPI app using Stable Diffusion API to generate images from user prompts. Have a look at code in the backend dir - start by installing the necessary dependencies listed in requirements.txt:

```bash
pip install -r requirements.txt
```

Have a look at the source code, specifically the code in routes/image_gen_router.

There are four different endpoints:

`POST /images/` - Generates an image based on a user prompt. The user prompt should be passed in the body of the request. Synchronous request responds with the image.

`POST /v2/image/` - Generates an image based on a user prompt. The user prompt should be passed in the body of the request. Asynchronous request using the fastAPI background_tasks api and responds with the image id.

`POST /v3/image/` - Generates an image based on a user prompt. The user prompt should be passed in the body of the request. Asynchronous request using a redis queue api and responds with the image id.

`GET /image/{image_id}` - Returns the image based on the image id. If the image is not yet generated it will return a 202.


Run the app inside the backend dir using uvicorn:

```bash
uvicorn app:app --reload
```

If you've postman installed you can now test all different endpoints using the following body:
    
```json
{
    "userPrompt": "<add your prompt here>"
}
```
   
If you don't have postman installed you can use curl to test the endpoints eg:

```bash
curl -X POST "http://localhost:8000/images/" -H  "accept: application/json" -H  "Content-Type: application/json" -d "{\"userPrompt\":\"<add your prompt here>\"}"
```


## Frontend App / Streamlit App

The Streamlit app is in `docker-lab/frontend/streamlit_app.py`. It sends HTTP requests to the backend from the Python process running Streamlit.

### Set `API_URL` before starting Streamlit

`API_URL` is the backend base URL, without an endpoint such as `/images` or a trailing slash. The frontend reads it when it starts.

For the first test, run both Python apps directly in the Codespace. Keep the backend running in another terminal. From the repository root:

```bash
cd docker-lab/frontend
python -m pip install -r requirements.txt
export API_URL=http://localhost:8000
streamlit run streamlit_app.py
```

`export` sets the variable for programs started from this terminal. Alternatively, set `API_URL=http://localhost:8000` in `docker-lab/frontend/.env`; the app loads that file when started from this directory. An already exported variable takes precedence. Restart Streamlit after changing the value.

In Codespaces, open port **8501** from the **Ports** tab to access the UI. The browser's forwarded URL is separate from `API_URL`, which is used by the Streamlit process to reach FastAPI.

| Where the apps run | Frontend `API_URL` |
|---|---|
| Both Python apps directly in the Codespace | `http://localhost:8000` |
| Containers named `backend` and `frontend` on the same custom network | `http://backend:8000` |
| Compose services named `backend` and `frontend` | `http://backend:8000` |

Inside the frontend container, `localhost` refers to that frontend container. Use the backend's network name when the apps run in separate containers.

Try **v1** and **v2**. The **v3** path needs Redis and a worker; you will add these in the optional Redis lab.


## (Start here -->) Dockerizing the backend app

### Minimum Python Version >= 3.11.

Your first task now is to dockerize the backend app. So, go to the backend dir, create an adequate Dockerfile, build it and run the container. 

Have a look at the Dockerfile reference from previous classes to see how to create a Dockerfile.

Ask yourself the following questions:

- What base image should I use?
- What files do I need to copy to the container?
- What commands do I need to run to install the requirements?
- What command do I need to run to start the app?
- What port do I need to expose?
- What command do I need to run to start the app?

After you've created the Dockerfile you can build the image using the following command:

A good image name could be `backend-img-generator`.

```bash
docker build -t backend-img-generator .
```

After the image is built you can run the image using the following command:

```bash
docker network create image-lab

docker run -d --name backend --network image-lab \
  --env-file .env -p 8000:8000 backend-img-generator
```

You can now test the app using curl or postman.

Run these commands from `docker-lab/backend`, where the backend `.env` file is located. Configure the container's startup command to listen on **`0.0.0.0:8000`**, for example `uvicorn app:app --host 0.0.0.0 --port 8000`. Binding only to `127.0.0.1` prevents other containers from reaching it.

For this lab, always give manually started containers explicit names with `--name`. Here, `backend` identifies the running container; `backend-img-generator` identifies the image used to create it. On the custom `image-lab` network, the container name also becomes its hostname.

Create the network once. If port 8000 is occupied by the earlier Python backend, stop that process before starting its container. If you are repeating the Docker step, remove only the previous lab container with `docker rm -f backend`, then run it again.


## Dockerizing the frontend app

Create a Dockerfile in `docker-lab/frontend`. Configure Streamlit to listen on `0.0.0.0:8501`. Build the image, then start a named container on the network created in the previous step.

Run from `docker-lab/frontend`:

```bash
docker build -t frontend-img-generator .
docker run -d --name frontend --network image-lab \
  -e API_URL=http://backend:8000 \
  -p 8501:8501 frontend-img-generator
```

Stop the earlier directly running Streamlit process first if it is still using port 8501.

`-e` passes an environment variable into the container. Setting `API_URL` in your Codespace terminal alone does not automatically pass it into a Docker container. Here, the explicit `-e` value also takes precedence over a `.env` file loaded by the app.

Both containers must join **`image-lab`**. The hostname **`backend`** in `API_URL` matches **`--name backend`**. Docker resolves that name on the custom network. Use port **8000**, the backend's container port, even if you publish it under a different host port. See [Docker's bridge networking guide](https://docs.docker.com/engine/network/drivers/bridge/).

### Check the connection

From the Codespace terminal:

```bash
docker ps
docker network inspect image-lab
docker exec frontend printenv API_URL
docker exec frontend python -c 'import os, requests; r = requests.get(os.environ["API_URL"] + "/openapi.json", timeout=10); print(r.status_code); r.raise_for_status()'
```

The variable should be `http://backend:8000`, and the HTTP check should print **200**. This checks connectivity from inside the frontend container. Then open port **8501** from the Codespaces Ports tab and try generating an image.

If it fails, check the names, network membership, and backend logs:

```bash
docker logs backend
docker logs frontend
```

To change a container's environment, recreate that container with the corrected `-e` value. `docker restart` keeps its existing environment. For example, run `docker rm -f frontend`, then repeat its `docker run` command above.

### Open a terminal or attach your editor

Names make it easy to identify the right container:

```bash
docker exec -it backend sh
docker exec -it frontend sh
```

Run one command at a time; use `exit` to return to the Codespace terminal. `docker exec` opens a separate shell while the application keeps running. Use `docker logs -f backend` to follow application output.

If using VS Code desktop with the Dev Containers extension and access to this Docker daemon, open the Command Palette and select **Dev Containers: Attach to Running Container...**, then choose `backend` or `frontend`. In the browser-based Codespace, use the terminal commands above. See [VS Code's attach guide](https://code.visualstudio.com/docs/devcontainers/attach-container).

Docker can assign random names, but explicit names make selecting, inspecting, and reconnecting to the same lab containers straightforward.

### Move on to Compose

Before starting the same apps through Compose, remove the two manually started lab containers to free their ports:

```bash
docker rm -f frontend backend
```

Now implement the Compose file below so Docker can create and connect both services for you.


## Docker Compose Integration

Now that you have successfully dockerized both the backend and frontend applications, the next task is to create and test a Docker Compose file. Docker Compose is a tool for defining and running multi-container Docker applications. With a single command, you can spin up both the frontend and backend services in a connected and consistent environment.

### Objective

Your task is to write a `docker-compose.yml` file that defines the configurations for both the backend and frontend services, ensuring they can communicate with each other effectively.

### Steps

1. **Create the Docker Compose File**: Complete `docker-lab/docker-compose.yml`. Run the Compose commands from `docker-lab/`.

2. **Define Services**:
   - Define two services: `backend` and `frontend`.
   - For each service, specify the Docker image to use (either using the images you built earlier or building them directly within the Docker Compose file).
   - Ensure the `frontend` service knows how to reach the `backend` service, typically through environment variables or Docker Compose's networking features.

3. **Set Up Networking**:
   - Configure Docker Compose networking to enable communication between the two services. By default, Docker Compose sets up a single network for all services in the `docker-compose.yml` file, and each service is reachable by other services at a hostname identical to the service name.

4. **Expose Ports**:
   - Expose the necessary ports for each service. For the backend, this might be port 8000, and for the frontend, it could be port 8501.

5. **Environment Configuration**:
   - Include any environment variables required by either service. Set the frontend's backend URL explicitly. Add this fragment under your `frontend` service:

     ```yaml
     environment:
       API_URL: http://backend:8000
     ```

   - `backend` is the Compose **service name**. Compose provides service-name DNS on its default network; you do not need to create `image-lab` manually for this step.
   - A Compose `.env` file supplies values for interpolation; it does not automatically inject every variable into containers. Use `environment` or `env_file` for container variables.
   - Compose already generates recognizable container names. Use `docker compose ps` to find them, `docker compose exec backend sh` to open a shell, and `docker compose logs -f backend` to follow logs. Explicit `container_name` is optional here; the API URL should still use the service name.
   - After changing the environment in Compose, run `docker compose up -d` to apply it.

6. **Volumes**:
   - If your applications require persistent storage or you want to use Docker volumes for live code reloading during development, define the necessary volumes.

### Testing the Docker Compose Setup

After writing your `docker-compose.yml` file, test it to ensure both services run correctly:

1. **Build and Run**:
   - Use `docker compose up --build` to build and start the services defined in your Docker Compose file.

2. **Verify Functionality**:
   - Check if the frontend is accessible at `http://localhost:8501` and is successfully communicating with the backend.
   - Test the backend independently to confirm it's functioning as expected.

3. **Debugging**:
   - If you encounter issues, use `docker compose logs` to check the logs of your services for debugging.
   - Make sure that the services are correctly configured and networked.



Reference: [Docker Compose networking](https://docs.docker.com/compose/how-tos/networking/).
